# Collections recovery: was the "+11% month-on-month" real?
**Approach.** Rebuild the headline from raw data, then test every candidate explanation (duplicates, time zones, attribution, mix, denominators, agents, vendors) with simple, transparent statistics.
All numbers below come from the DuckDB database built by `python src/run_pipeline.py` (SQL in `sql/`). Every conclusion carries a label: **Fact / Strong evidence / Correlation / Hypothesis**.

**Bottom line (verified in the cells below):** the +11% is a **calendar artefact**. It is March's 31 days vs February's 28 days (+10.7%) plus a small duplicate/reversal effect. Per calendar day, recovery changed by about **-0.1%** (p = 0.98). No operational driver we can measure moves recovery or contact rate.

In [1]:
import os, duckdb, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
os.chdir('..') if os.path.basename(os.getcwd())=='notebooks' else None
os.makedirs('outputs/figures',exist_ok=True); os.makedirs('outputs/tables',exist_ok=True)
con = duckdb.connect('outputs/collections.duckdb', read_only=True)
q = lambda sql: con.sql(sql).df()
pd.set_option('display.width',200,'display.max_columns',30,'display.float_format',lambda x:f'{x:,.4f}')
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False,'font.size':9})
RAW = {t: con.execute(f'select count(*) from raw.{t}').fetchall()[0][0] for t in [r[0] for r in con.execute("select table_name from information_schema.tables where table_schema='raw'").fetchall()]}
pd.Series(RAW,name='raw_rows').sort_values(ascending=False).to_frame()

,raw_rows
call_attempts,120000
calls,91350
whatsapp_events,60600
account_status_history,60000
daily_targeting,45000
sms_events,45000
call_dispositions,35000
borrowers,30600
accounts,30000
agents,30000


## 1. What happened
### 1a. Reproduce the headline
Only one construction of the data gives +11%: **sum of every `SUCCESS` payment row (duplicates included), by calendar month, March vs February**. That is almost certainly the reported number.

In [2]:
mom = q("select * from metrics.mom"); mom.to_csv('outputs/tables/mom.csv',index=False)
view = mom[['month','reported_success','reported_mom','net_recovery','golden_net_mom','golden_per_day_mom','is_partial_month']].copy()
view

,month,reported_success,reported_mom,net_recovery,golden_net_mom,golden_per_day_mom,is_partial_month
0,2026-01,"191,133,284.4200",NaN,"175,282,985.0300",NaN,NaN,False
1,2026-02,"174,097,287.9600",-0.0891,"158,197,094.8400",-0.0975,-0.0008,False
2,2026-03,"193,233,384.2900",0.1099,"174,935,389.9900",0.1058,-0.0012,False
3,2026-04,"178,427,017.0500",-0.0766,"160,944,491.9800",-0.0800,-0.0493,False
4,2026-05,"187,048,144.3400",0.0483,"171,875,388.9300",0.0679,0.0335,False
5,2026-06,"178,724,493.4600",-0.0445,"162,849,607.5200",-0.0525,-0.0209,False
6,2026-07,"190,278,846.8800",0.0646,"171,878,796.9400",0.0554,0.0214,False
7,2026-08,"48,543,467.9300",-0.7449,"44,953,643.5000",-0.7385,-0.0991,True


Reading the table: `reported_mom` for 2026-03 is **+10.99%**. After de-duplication and netting reversals (`golden_net_mom`) it is still +10.6%, so cleaning explains only ~0.4 pts. The big move is the next column: dividing by days in the month (`golden_per_day_mom`) gives **-0.12%**.
August is a partial month (data stops 8-Aug) and must be excluded from any MoM; the raw series shows it as -74%.

In [3]:
fig,ax=plt.subplots(1,2,figsize=(11,3.6))
m=mom[~mom.is_partial_month].reset_index(drop=True)
w=.38;x=np.arange(len(m))
ax[0].bar(x-w/2,m.reported_mom.fillna(0)*100,w,label='Reported (raw SUCCESS, calendar month)',color='#c0504d')
ax[0].bar(x+w/2,m.golden_per_day_mom.fillna(0)*100,w,label='Golden (deduped, net, per day)',color='#1f4e79')
ax[0].set_xticks(x);ax[0].set_xticklabels(m.month.str[5:]);ax[0].axhline(0,c='k',lw=.6);ax[0].set_ylabel('MoM %');ax[0].legend(frameon=False,fontsize=7);ax[0].set_title('Month-on-month change')
dly=q("select event_date_ist d,sum(net_amount)/1e6 net from golden.fact_payment where event_date_ist<'2026-08-01' group by 1 order by 1");dly['d']=pd.to_datetime(dly.d)
ax[1].plot(dly.d,dly.net,lw=.6,c='#999');ax[1].plot(dly.d,dly.net.rolling(14).mean(),c='#1f4e79',lw=1.8,label='14-day mean');ax[1].set_title('Daily net recovery (Rs M): flat');ax[1].legend(frameon=False)
plt.tight_layout();plt.savefig('outputs/figures/headline_vs_golden.png');plt.show()

C:\Users\dell\AppData\Local\Temp\ipykernel_9244\2627280911.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.savefig('outputs/figures/headline_vs_golden.png');plt.show()


### 1b. Is any change statistically distinguishable from noise?
Daily net recovery has a coefficient of variation of ~15%, so a single month's mean carries ~±2.7% noise. Test Feb vs Mar per-day, and a linear trend over all 212 days.

In [4]:
d=q("select event_date_ist d,sum(net_amount) net from golden.fact_payment where event_date_ist<'2026-08-01' group by 1 order by 1");d['d']=pd.to_datetime(d.d);d['m']=d.d.dt.strftime('%Y-%m')
f,m_=d[d.m=='2026-02'].net,d[d.m=='2026-03'].net
t=stats.ttest_ind(m_,f,equal_var=False);rng=np.random.default_rng(1)
bs=[rng.choice(m_,len(m_)).mean()/rng.choice(f,len(f)).mean()-1 for _ in range(5000)]
sl=stats.linregress(np.arange(len(d)),d.net)
pd.Series({'Feb mean/day (Rs)':f.mean(),'Mar mean/day (Rs)':m_.mean(),'per-day change':m_.mean()/f.mean()-1,'Welch t-test p':t.pvalue,'bootstrap 95% CI low':np.percentile(bs,2.5),'bootstrap 95% CI high':np.percentile(bs,97.5),
 'linear trend, % of mean over period':sl.slope*len(d)/d.net.mean(),'trend p-value':sl.pvalue,'daily CV':d.net.std()/d.net.mean()}).to_frame('value')

,value
Feb mean/day (Rs),"5,649,896.2443"
Mar mean/day (Rs),"5,643,077.0965"
per-day change,-0.0012
Welch t-test p,0.9753
bootstrap 95% CI low,-0.0727
bootstrap 95% CI high,0.0773
"linear trend, % of mean over period",-0.0382
trend p-value,0.2855
daily CV,0.1500


**Fact:** per-day recovery Feb to Mar changed by ~-0.1% (95% CI about -7% to +8%). **Strong evidence:** there is no trend across Jan-Jul (p = 0.29). The only month that looks different is April (-4.9% per day), and that is within normal month-to-month noise.

## 2. Data forensics (assignment Part 2, items A to G)
### Raw to Rejected/Corrected to Golden

In [5]:
wf=q("select ord,dataset,step,n,amount from metrics.dq_waterfall order by ord");wf.to_csv('outputs/tables/dq_waterfall.csv',index=False);wf

,ord,dataset,step,n,amount
0,1,payments,raw rows,25500,"1,917,258,617.1500"
1,2,payments,rejected: duplicate payment_id,500,"38,366,344.2600"
2,3,payments,golden rows,25000,"1,878,892,272.8900"
3,4,payments,of which SUCCESS (gross recovery),17534,"1,315,583,964.6400"
4,5,payments,of which REVERSED (netted off),1254,"94,666,565.9100"
5,6,payments,"of which PENDING (excluded, unconfirmed)",2535,"190,223,176.3700"
6,7,payments,of which FAILED (excluded),3677,"278,418,565.9700"
7,8,payments,"corrected: missing payment_reference (kept, fl...",371,"28,007,300.5500"
8,9,payments,corrected: month changes after IST normalisation,107,"7,641,704.5200"
9,10,calls,raw rows,91350,NaN


**A. Duplicate payments - Fact.** 500 payment rows are exact replays (same `payment_id`, timestamp, amount). They carry ₹3.84 Cr (all statuses); the SUCCESS-status subset is shown below. Do NOT dedupe on `payment_reference`: 3,406 references are shared by different accounts and amounts, so a reference-based dedupe would delete ~3,800 genuine payments.

In [6]:
dup=q('''select s.payment_status status,count(*) rows_rejected,sum(s.amount)/1e6 amt_m from stg.payments s join clean.rejects x on x.src='payments' and x._raw_row=s._raw_row group by 1 order by 1''')
ref=q('''select count(*) refs_shared,sum(n) rows_in_shared_refs from (select payment_reference,count(*) n from clean.payments where payment_reference is not null group by 1 having count(distinct account_id)>1)''')
same_acct=q('''select count(*) pairs_same_account_and_amount from clean.payments a join clean.payments b on a.payment_reference=b.payment_reference and a.payment_id<b.payment_id and a.account_id=b.account_id and a.amount=b.amount''')
print(dup.to_string(index=False));print(ref.to_string(index=False));print(same_acct.to_string(index=False))
inflation=q("select sum(case when payment_status='SUCCESS' then amount end) from raw.payments").iloc[0,0]/q("select sum(amount) from golden.fact_payment where payment_status='SUCCESS'").iloc[0,0]-1
print('Duplicates inflate gross SUCCESS recovery by',round(inflation*100,2),'%  (roughly flat across months, so they do not create a trend)')
print(q("select strftime(s.event_at,'%Y-%m') m,count(*) dups from stg.payments s join clean.rejects x on x.src='payments' and x._raw_row=s._raw_row group by 1 order by 1").T.to_string())

  status  rows_rejected   amt_m
  FAILED             67  5.0878
 PENDING             57  4.6448
REVERSED             30  2.7318
 SUCCESS            346 25.9020
 refs_shared  rows_in_shared_refs
        3406           7,216.0000
 pairs_same_account_and_amount
                             0
Duplicates inflate gross SUCCESS recovery by 1.97 %  (roughly flat across months, so they do not create a trend)
            0        1        2        3        4        5        6        7
m     2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07  2026-08
dups       75       67       73       70       67       58       62       28


**B. Attribution errors - Strong evidence.** Crediting an account's *latest campaign* (a common shortcut) assigns payments to campaigns that **started after the payment** in about a third of cases. The proper rule is last touch strictly before the payment, inside a window. The window changes the story completely (below).

In [7]:
rows=[]
for w in [1,3,7,14,30,60,10000]:
    r=q(f"select count(*) filter(where lag_days<={w})*1.0/count(*) share_pay, sum(amount) filter(where lag_days<={w})/sum(amount) share_amt from golden.payment_attribution").iloc[0]
    rows.append({'window_days':w if w<10000 else 'any prior touch','share_of_payments_attributed':r.share_pay,'share_of_rupees':r.share_amt})
att=pd.DataFrame(rows);att.to_csv('outputs/tables/attribution_window.csv',index=False)
late=q('''select count(*) n,count(*) filter(where cm.start_at>p.event_ts_ist)*1.0/count(*) latest_campaign_started_after_payment from golden.payment_attribution p left join golden.latest_campaign_by_account l using(account_id) left join raw.campaigns cm on cm.campaign_id=l.latest_campaign''')
print(att.to_string(index=False));print(late.to_string(index=False))
inw=q("select count(*) filter(where f.event_ts_ist between c.start_at and c.end_at)*1.0/count(*) inside_window from golden.fact_call f join raw.campaigns c using(campaign_id)");print('calls dated inside their own campaign start/end window:',round(inw.iloc[0,0],3))

    window_days  share_of_payments_attributed  share_of_rupees
              1                        0.0295           0.0296
              3                        0.0828           0.0832
              7                        0.1781           0.1792
             14                        0.3218           0.3251
             30                        0.5490           0.5516
             60                        0.7456           0.7459
any prior touch                        0.8398           0.8405
    n  latest_campaign_started_after_payment
17534                                 0.3294
calls dated inside their own campaign start/end window: 0.194


Only **18% of payments have any touch in the prior 7 days** and 55% within 30. Attribution is therefore extremely window-sensitive: a 30-day window credits 3x the recovery a 7-day window does. Also, only ~19% of calls fall inside the start/end window of the campaign they are tagged with (**inconsistent campaign definitions - Fact**), so campaign-level ROI cannot be trusted from the campaign tag alone.

**C. Time-zone problems - Fact.** `calls.timezone` mixes UTC / Asia/Kolkata / Asia/Dubai (one third each). Converting all to IST changes the date of ~10% of calls (8,791), the month of 305 calls, and the hour of 67% (59,962). Payments have no timezone column; we used the provider's vendor timezone (assumption, tested below).

In [8]:
tz=q('''select count(*) calls, count(*) filter(where event_date_ist<>cast(event_at_src as date)) date_changed, count(*) filter(where month_ist<>month_src) month_changed, count(*) filter(where hour_ist<>hour_src) hour_changed,
 count(*) filter(where hour_src between 8 and 18)*1.0/count(*) in_window_src, count(*) filter(where hour_ist between 8 and 18)*1.0/count(*) in_window_ist from golden.fact_call where direction='OUTBOUND' ''');print(tz.T.to_string())
alt=q("select month_ist m,sum(net_amount) a from golden.fact_payment group by 1 order by 1");alt2=q("select strftime(event_at_src,'%Y-%m') m,sum(net_amount) a from golden.fact_payment group by 1 order by 1")
x=alt.merge(alt2,on='m');x['diff_pct']=x.a_x/x.a_y-1;print('Payment tz sensitivity (IST-normalised vs source-clock monthly net):');print(x.to_string(index=False))

                        0
calls         82,907.0000
date_changed   8,111.0000
month_changed    275.0000
hour_changed  55,279.0000
in_window_src      0.4570
in_window_ist      0.4587
Payment tz sensitivity (IST-normalised vs source-clock monthly net):
      m              a_x              a_y  diff_pct
2026-01 175,282,985.0300 175,597,902.5300   -0.0018
2026-02 158,197,094.8400 158,664,771.0900   -0.0029
2026-03 174,935,389.9900 174,547,919.1700    0.0022
2026-04 160,944,491.9800 161,810,682.0000   -0.0054
2026-05 171,875,388.9300 171,559,358.0600    0.0018
2026-06 162,849,607.5200 162,610,249.6200    0.0015
2026-07 171,878,796.9400 171,975,871.6200   -0.0006
2026-08  44,953,643.5000  44,150,644.6400    0.0182


Compliance angle (**Fact, subject to timestamps being correct**): about **54% of outbound calls fall outside 08:00-19:00 IST**, the window generally applied to borrower calls. This is independent of the recovery question but should be escalated.

**D. Vendor / disposition mapping - Fact (no drift found).** Three disposition schema versions (legacy, v1, v2) coexist in every month at ~33% each, and the legacy code `PTP` is the same concept as `PROMISE_TO_PAY` (3,904 rows). The *mix* of codes does not change over time (below), so code changes cannot manufacture a trend. They do split any PTP count in two if left unmapped: **`PTP` + `PROMISE_TO_PAY` must be summed.** Vendor ids also do not change meaning over time (all 15 appear in every month).

In [9]:
dd=q("select strftime(event_ts_ist,'%Y-%m') m,disposition_raw c,count(*) n from clean.dispositions where strftime(event_ts_ist,'%Y-%m')<'2026-08' group by 1,2")
t=dd.pivot(index='m',columns='c',values='n');print('chi2 p-value, code mix homogeneous across months:',round(stats.chi2_contingency(t)[1],3))
cov=q('''select count(*) outbound_calls, count(disposition_norm)*1.0/count(*) share_with_disposition,
 count(*) filter(where not is_answered and disposition_norm in ('PROMISE_TO_PAY','PAID'))  unanswered_but_PTP_or_PAID from golden.fact_call where direction='OUTBOUND' ''');print(cov.T.to_string())

chi2 p-value, code mix homogeneous across months: 0.879


                                     0
outbound_calls             82,907.0000
share_with_disposition          0.3223
unanswered_but_PTP_or_PAID  7,136.0000


Also **Fact:** only ~32% of outbound calls have a disposition, and 7,136 *unanswered* calls carry a `PROMISE_TO_PAY`/`PAID` disposition. `call_attempts.attempt_status` and `calls.call_status` are statistically independent (a call marked ANSWERED is as likely to have a FAILED attempt as a CONNECTED one). We therefore use `calls.call_status` as the single source of truth for contact, and re-key attempts and dispositions to their parent call via `call_id` (their own `account_id` and `agent_id` disagree with the parent call in ~100% of rows).

**E. Agent identity - Fact.** 30,000 agent rows describe 1,000 `agent_id`s (29.6 versions each). `employee_code` has 1,099 values, each mapped to ~27 different agent_ids, and there are only 10 distinct names. So no alternate key can resolve identity: `agent_id` is the only key facts use, so we use it and take the latest version per id. 975 agent_ids have at least one version with `updated_at` earlier than `joined_at`, so tenure is a low-confidence field (**Hypothesis** level for any tenure finding). 1,759 calls (2.0%) have no agent at all; 68 were repaired from a sibling duplicate.

**F. Portfolio mix - Strong evidence of no shift.** Recovery composition by DPD bucket, loan type and risk segment is statistically stable across months (tests below). **G. Denominator manipulation** is covered in section 3.

## 3. Statistical investigation (Part 3)
### 3a. Mix effects, Simpson's paradox, cohort effects

In [10]:
res=[]
for col in ['dpd_bucket','loan_type','risk_segment']:
    cnt=q(f"select month_ist m,{col} g,count(*) n from golden.fact_payment where payment_status='SUCCESS' and month_ist<'2026-08' group by 1,2").pivot(index='m',columns='g',values='n')
    res.append({'dimension':col,'test':'chi2: payer mix same across months','p_value':stats.chi2_contingency(cnt)[1]})
coh=q("select strftime(opened_at,'%Y-%m') v,count(*) n from golden.dim_account group by 1 order by 1");print('account vintages:',coh.v.min(),'to',coh.v.max())
cnt=q("select month_ist m,strftime(a.opened_at,'%Y-%m') v,count(*) n from golden.fact_payment p join golden.dim_account a using(account_id) where payment_status='SUCCESS' and month_ist<'2026-08' group by 1,2").pivot(index='m',columns='v',values='n').fillna(0)
res.append({'dimension':'vintage (opened_at month)','test':'chi2: payer vintage mix same across months','p_value':stats.chi2_contingency(cnt)[1]})
pd.DataFrame(res)

account vintages: 2024-01 to 2025-11


,dimension,test,p_value
0,dpd_bucket,chi2: payer mix same across months,0.6869
1,loan_type,chi2: payer mix same across months,0.9977
2,risk_segment,chi2: payer mix same across months,0.2300
3,vintage (opened_at month),chi2: payer vintage mix same across months,0.0963


In [11]:
# Shift-share (Kitagawa) decomposition of Feb->Mar change in recovery per touched account, by DPD bucket
x=q('''with touched as (select distinct account_id, strftime(touch_ts,'%Y-%m') m from golden.fact_touch),
pay as (select account_id,month_ist m,sum(net_amount) r from golden.fact_payment group by 1,2)
select t.m, a.dpd_bucket seg, count(*) accts, coalesce(sum(p.r),0) rec from touched t join golden.dim_account a using(account_id) left join pay p on p.account_id=t.account_id and p.m=t.m
where t.m in ('2026-02','2026-03') group by 1,2''')
f=x[x.m=='2026-02'].set_index('seg');m=x[x.m=='2026-03'].set_index('seg')
w0,w1=f.accts/f.accts.sum(),m.accts/m.accts.sum();rf,rm=f.rec/f.accts,m.rec/m.accts
tot=(w1*rm).sum()-(w0*rf).sum();mix=((w1-w0)*(rf+rm)/2).sum();rate=(((w0+w1)/2)*(rm-rf)).sum()
print(pd.Series({'recovery per touched account Feb':(w0*rf).sum(),'Mar':(w1*rm).sum(),'total change':tot,'mix effect':mix,'rate effect':rate,'mix share of change':mix/tot}).to_string())

recovery per touched account Feb   5,372.3999
Mar                                5,977.6001
total change                         605.2002
mix effect                             0.4270
rate effect                          604.7732
mix share of change                    0.0007


**Strong evidence:** payer mix by DPD, loan type, risk segment and account vintage does not differ across months (all p ≥ 0.09, none significant), and the mix term in the Feb-to-Mar decomposition is small relative to the rate term. A portfolio acquired in the period would show a vintage shift; none is visible. (`accounts` is a static snapshot, so exposure mix cannot change over time. A genuine portfolio-acquisition test needs dated balances; see the recommendation.)

**Simpson's paradox.** If the aggregate +11% hid opposing sub-group moves, the raw per-segment MoM would disagree in sign. Check it:

In [12]:
rows=[]
for col in ['dpd_bucket','loan_type','risk_segment']:
    t=q(f"select {col} g,month_ist m,sum(net_amount) s from golden.fact_payment where month_ist in ('2026-02','2026-03') group by 1,2").pivot(index='g',columns='m',values='s')
    raw=q(f"select {col} g,strftime(event_at_src,'%Y-%m') m,sum(amount) s from golden.fact_payment where payment_status='SUCCESS' and strftime(event_at_src,'%Y-%m') in ('2026-02','2026-03') group by 1,2").pivot(index='g',columns='m',values='s')
    for g in t.index: rows.append({'dim':col,'segment':g,'raw_MoM':raw.loc[g,'2026-03']/raw.loc[g,'2026-02']-1,'per_day_MoM':(t.loc[g,'2026-03']/31)/(t.loc[g,'2026-02']/28)-1})
sp=pd.DataFrame(rows);print(sp.groupby('dim')[['raw_MoM','per_day_MoM']].agg(['min','max']).to_string())
print('segments with raw MoM > 0:',(sp.raw_MoM>0).sum(),'of',len(sp),'| segments with per-day MoM > 0:',(sp.per_day_MoM>0).sum(),'of',len(sp))

             raw_MoM        per_day_MoM       
                 min    max         min    max
dim                                           
dpd_bucket    0.0048 0.2459     -0.1156 0.1448
loan_type     0.0711 0.1410     -0.0487 0.0349
risk_segment  0.0392 0.1568     -0.0319 0.0120
segments with raw MoM > 0: 14 of 14 | segments with per-day MoM > 0: 9 of 14


The raw gain is positive in **all 14 of 14** segments (loan type +7% to +14%, risk segment +4% to +16%, DPD bucket +0.5% to +25%): the signature of a uniform calendar effect; a mix effect would be concentrated in a few segments. No Simpson reversal. Per day, only 9 of 14 segments are positive and they scatter around zero (-12% to +14%): noise.

### 3b. Denominator manipulation, survivorship and selection bias

In [13]:
st=q('''with e as (select acct_status_snapshot s,count(*) n,sum(outstanding_amount) ex from golden.dim_account group by 1),
p as (select acct_status_snapshot s,sum(net_amount) net,count(*) filter(where payment_status='SUCCESS') k from golden.fact_payment group by 1)
select s status,n accounts,ex/1e9 exposure_bn,net/1e6 net_rec_m,net/ex rec_rate,k successful_payments from e join p using(s)''');print(st.to_string(index=False))
print('share of successful payments landing on accounts whose snapshot status is NOT ACTIVE:',round(1-st.set_index('status').loc['ACTIVE','successful_payments']/st.successful_payments.sum(),3))
tg=q('''select t.status,count(*) n,avg((exists(select 1 from golden.fact_payment p where p.account_id=t.account_id and p.payment_status='SUCCESS' and p.event_ts_ist between t.target_date and t.target_date+interval 14 day))::int) paid_within_14d from raw.daily_targeting t group by 1 order by 1''');print(tg.to_string(index=False))

  status  accounts  exposure_bn  net_rec_m  rec_rate  successful_payments
WRITEOFF      7479       2.5980   303.4860    0.1168                 4370
  ACTIVE      7539       2.6374   311.1587    0.1180                 4404
    PAID      7486       2.6175   303.2839    0.1159                 4379
  CLOSED      7496       2.6361   302.9889    0.1149                 4381
share of successful payments landing on accounts whose snapshot status is NOT ACTIVE: 0.749
   status     n  paid_within_14d
CONTACTED 11254           0.0353
  EXPIRED 11371           0.0363
   QUEUED 11202           0.0365
  SKIPPED 11173           0.0363


**Fact:** `accounts.status` is a current snapshot that is unrelated to payment behaviour. **75% of successful payments sit on accounts marked CLOSED, PAID or WRITEOFF** with identical rates across all four statuses. So a denominator of "ACTIVE accounts only" would silently drop 75% of paying accounts from the base: it is survivorship bias. We use *all accounts* as the denominator and never filter on snapshot status.

**Selection bias in targeting - Fact:** accounts the targeting engine `SKIPPED`/`EXPIRED` pay within 14 days at the same rate (≈3.6%) as those `CONTACTED`. Either targeting has no value or the status field is not a reliable record of what happened; either way "contacted accounts convert better" is not supported. Another way unsuccessful accounts vanish: **PTPs** (below).

### 3c. PTP and other conversion metrics, reported vs independent

In [14]:
p=q("select status,count(*) n,avg(verified_paid::int) verified_rate,sum(promised_amount)/1e6 promised_m from golden.fact_ptp group by 1 order by 1");print(p.to_string(index=False))
k=p.set_index('status');print('Claimed PTP kept rate K/(K+B):',round(k.loc['KEPT','n']/(k.loc['KEPT','n']+k.loc['BROKEN','n']),3))
print('Verified (a SUCCESS payment on the account by promised_date+3d) among KEPT:',round(k.loc['KEPT','verified_rate'],3),'| among BROKEN:',round(k.loc['BROKEN','verified_rate'],3))
base=q("select avg(exists_pay::int) from (select (exists(select 1 from clean.payments pay where pay.account_id=a.account_id and pay.payment_status='SUCCESS' and pay.event_ts_ist between timestamp '2026-03-01' and timestamp '2026-03-18'))::int exists_pay from golden.dim_account a)").iloc[0,0];print('Random-account 18-day payment rate (coincidence baseline):',round(base,3))
t=stats.chi2_contingency(pd.crosstab(q("select status,verified_paid from golden.fact_ptp").status.isin(['KEPT']),q("select status,verified_paid from golden.fact_ptp").verified_paid))[1];print('chi2 p (KEPT vs rest verified):',round(t,3))

   status    n  verified_rate  promised_m
   BROKEN 4553         0.0409    228.7966
CANCELLED 4543         0.0431    228.5406
     KEPT 4489         0.0479    224.9404
     OPEN 4415         0.0412    221.9315
Claimed PTP kept rate K/(K+B): 0.496
Verified (a SUCCESS payment on the account by promised_date+3d) among KEPT: 0.048 | among BROKEN: 0.041
Random-account 18-day payment rate (coincidence baseline): 0.046
chi2 p (KEPT vs rest verified): 0.087


**Fact:** the claimed PTP kept rate (~50%) is not corroborated by payments: only ~4.8% of `KEPT` promises have a successful payment on that account by the promised date +3 days, which is no better than BROKEN promises (~4.1%) or coincidence. So PTP status is **not usable as a recovery proxy** and PTP kept rate is a vanity metric here until the status is reconciled to payment events.

### 3d. Drivers: do contact rates differ by anything? (with multiple-testing control)
We test 17 dimensions. Expect ~1 false positive at α = 0.05 by chance alone, so use Bonferroni (0.05/17 = 0.003).

In [15]:
fc=q("select f.*,c.channel camp_channel,c.campaign_name,c.strategy_version,c.target_definition,vt.vendor_name from golden.fact_call f left join raw.campaigns c using(campaign_id) left join raw.vendor_telephony vt on vt.vendor_id=f.vendor_id where direction='OUTBOUND' and month_ist>='2026-01'")
rows=[]
for col in ['month_ist','vendor_id','vendor_name','time_band_ist','hour_src','tenure_bucket','attempt_no','camp_channel','campaign_name','strategy_version','target_definition','dpd_bucket','loan_type','risk_segment','state','team','src_tz']:
    g=fc.groupby(col).is_answered.mean();p=stats.chi2_contingency(pd.crosstab(fc[col],fc.is_answered))[1]
    rows.append({'dimension':col,'groups':len(g),'min_contact_rate':g.min(),'max_contact_rate':g.max(),'chi2_p':p,'significant_after_Bonferroni':p<0.05/17})
dr=pd.DataFrame(rows).sort_values('chi2_p');dr.to_csv('outputs/tables/driver_scan.csv',index=False);dr

,dimension,groups,min_contact_rate,max_contact_rate,chi2_p,significant_after_Bonferroni
14,state,10,0.1877,0.2097,0.0243,False
2,vendor_name,5,0.1949,0.2068,0.0497,False
3,time_band_ist,5,0.1904,0.2044,0.0567,False
13,risk_segment,4,0.1955,0.2031,0.2572,False
0,month_ist,8,0.1932,0.2047,0.2939,False
10,target_definition,5,0.1958,0.2048,0.3041,False
8,campaign_name,5,0.1942,0.2027,0.3169,False
4,hour_src,24,0.1813,0.2091,0.3656,False
12,loan_type,5,0.1954,0.2033,0.3949,False
7,camp_channel,5,0.1944,0.2025,0.4104,False


In [16]:
a=q("select agent_id,count(*) n,sum(is_answered::int) k from golden.fact_call where direction='OUTBOUND' and agent_id is not null group by 1");pp=a.k.sum()/a.n.sum()
chi=(((a.k-a.n*pp)**2)/(a.n*pp*(1-pp))).sum();print(f'Agents: {len(a)}, ~{a.n.mean():.0f} calls each. Dispersion ratio = {chi/(len(a)-1):.3f} (1.0 = pure chance), p = {1-stats.chi2.cdf(chi,len(a)-1):.2f}')
r=a.assign(r=a.k/a.n).r;print('Best/worst decile of agents by contact rate:',round(r.quantile(.9),3),'/',round(r.quantile(.1),3),'-> exactly the spread binomial noise produces')
d=q('''with p as (select event_date_ist d,sum(net_amount)/1e6 net from golden.fact_payment group by 1),cl as (select event_date_ist d,count(*)/100.0 dials from golden.fact_call where direction='OUTBOUND' group by 1),
s as (select cast(login_ist as date) d,sum(hours)/100 hrs from clean.sessions group by 1),w as (select cast(event_at as date) d,count(*)/100.0 wa from raw.whatsapp_events group by 1),
m as (select cast(event_at as date) d,count(*)/100.0 sms from raw.sms_events group by 1),f as (select cast(event_at as date) d,count(*)/100.0 fv from raw.field_visits group by 1)
select * from p join cl using(d) join s using(d) join w using(d) join m using(d) join f using(d) where d between '2026-01-02' and '2026-07-31' ''');d['dow']=pd.to_datetime(d.d).dt.dayofweek
r=smf.ols('net~hrs+dials+wa+sms+fv+C(dow)',data=d).fit(cov_type='HAC',cov_kwds={'maxlags':3})
ci=r.conf_int().loc[['hrs','dials','wa','sms','fv']];ci.columns=['ci_low','ci_high'];ci['coef_Rs_M_per_100_units']=r.params[['hrs','dials','wa','sms','fv']];ci['p']=r.pvalues[['hrs','dials','wa','sms','fv']]
print('Daily recovery vs daily inputs (Rs M per +100 units), HAC SEs, n =',len(d),', R2 =',round(r.rsquared,3),', joint F p =',round(r.f_pvalue,2));ci.to_csv('outputs/tables/input_elasticity.csv')
ci

Agents: 1000, ~81 calls each. Dispersion ratio = 0.987 (1.0 = pure chance), p = 0.61
Best/worst decile of agents by contact rate: 0.256 / 0.143 -> exactly the spread binomial noise produces
Daily recovery vs daily inputs (Rs M per +100 units), HAC SEs, n = 211 , R2 = 0.041 , joint F p = 0.26


C:\Users\dell\AppData\Local\Temp\ipykernel_9244\943789022.py:7: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  select * from p join cl using(d) join s using(d) join w using(d) join m using(d) join f using(d) where d between '2026-01-02' and '2026-07-31' ''');d['dow']=pd.to_datetime(d.d).dt.dayofweek


,ci_low,ci_high,coef_Rs_M_per_100_units,p
hrs,-0.3305,0.1928,-0.0689,0.6060
dials,-0.3321,0.8076,0.2377,0.4135
wa,-1.2193,0.0451,-0.5871,0.0687
sms,-0.4583,1.0688,0.3053,0.4333
fv,-0.7388,1.6308,0.4460,0.4606


**Findings.** Fact-level: no dimension is significant after Bonferroni. The raw p-values for `state` (0.024), `vendor_name` (0.050) and `time_band` (0.057) are the 1-in-17 false positives you expect. Agent-to-agent variation is indistinguishable from binomial noise (dispersion 0.99), so there is no "top agent" signal to scale and no tenure effect (**Strong evidence** of absence, with tenure itself **Hypothesis**-level because of the joined_at conflicts). Daily recovery does not respond to daily agent-hours, dials, WhatsApp, SMS or field volume (joint p = 0.26, R² = 4%). **Language** is not in the data at all: `state` is the only proxy, and it shows no effect (**Hypothesis** at best).

### 3e. Time-series effects
Daily recovery has a mild weekday pattern (Saturday ~4% lower) and ~15% daily noise, but no trend and no change point. A 28-day February against a 31-day March is the single largest source of apparent movement.

## 4. Independent metric definitions
Reported-style vs golden (details and rationale in `docs/metric_definitions.md`).

In [17]:
mm=q("select * from metrics.monthly where not is_partial_month")
cols=['month','success_amt','net_recovery','net_recovery_per_day','recovery_rate_30d_norm','contact_rate','account_contact_rate','agent_hours','recovery_per_agent_hour','assumed_cost_inr','cost_per_rupee_recovered']
t=mm[cols].copy();t.to_csv('outputs/tables/monthly_scorecard.csv',index=False);t

,month,success_amt,net_recovery,net_recovery_per_day,recovery_rate_30d_norm,contact_rate,account_contact_rate,agent_hours,recovery_per_agent_hour,assumed_cost_inr,cost_per_rupee_recovered
0,2026-01,"186,892,948.0900","175,282,985.0300","5,654,289.8397",0.0162,0.2007,0.2333,"11,113.7803","15,771.6799","2,923,503.0400",0.0167
1,2026-02,"169,696,039.5100","158,197,094.8400","5,649,896.2443",0.0162,0.1979,0.2277,"10,532.8147","15,019.4510","2,749,047.3300",0.0174
2,2026-03,"189,148,499.6400","174,935,389.9900","5,643,077.0965",0.0161,0.1997,0.2331,"11,152.6914","15,685.4865","2,958,871.7700",0.0169
3,2026-04,"174,423,198.5900","160,944,491.9800","5,364,816.3993",0.0153,0.1935,0.2235,"10,629.9728","15,140.6307","2,834,828.5700",0.0176
4,2026-05,"184,566,309.3700","171,875,388.9300","5,544,367.3848",0.0159,0.2032,0.2363,"10,841.5914","15,853.3358","2,895,167.8500",0.0168
5,2026-06,"175,799,084.8700","162,849,607.5200","5,428,320.2507",0.0155,0.2047,0.2378,"10,682.2358","15,244.8991","2,818,066.4000",0.0173
6,2026-07,"187,064,406.0500","171,878,796.9400","5,544,477.3206",0.0159,0.1940,0.2260,"11,210.8597","15,331.4555","2,966,123.2500",0.0173


In [18]:
rpc=q('''select count(*) answered, count(disposition_norm) with_disp, count(*) filter(where disposition_norm not in ('WRONG_NUMBER','NO_CONTACT')) rpc_disp from golden.fact_call where direction='OUTBOUND' and is_answered''')
ptp_calls=q("select count(*) filter(where disposition_norm='PROMISE_TO_PAY') ptp, count(*) filter(where disposition_norm is not null) disp from golden.fact_call where direction='OUTBOUND'")
print(rpc.T.to_string()); print('RPC rate (answered with a right-party disposition / answered):',round(rpc.rpc_disp[0]/rpc.answered[0],3),' <- only computable for the',round(rpc.with_disp[0]/rpc.answered[0],2),'of answered calls that have any disposition; the rest are missing, so RPC is a lower bound.')
print('PTP rate among dispositioned calls (PTP+PROMISE_TO_PAY):',round(ptp_calls.ptp[0]/ptp_calls.disp[0],3))

               0
answered   16490
with_disp   5340
rpc_disp    4165
RPC rate (answered with a right-party disposition / answered): 0.253  <- only computable for the 0.32 of answered calls that have any disposition; the rest are missing, so RPC is a lower bound.
PTP rate among dispositioned calls (PTP+PROMISE_TO_PAY): 0.224


**What changes when definitions are rebuilt** (all verified above):
- *Recovery* = SUCCESS minus REVERSED, deduplicated by `payment_id`, IST months, normalised per calendar day. Reported MoM +11.0% → -0.1%.
- *Contact rate* = ANSWERED outbound / outbound dials, stable at 19.4-20.5% every month (no improvement). Account-level contact rate 22.3-23.8%.
- *RPC*, *PTP rate*, *PTP kept rate* are computable only for dispositioned calls (32%) and are unverifiable against payments. Report them with coverage; do not headline them.
- *Recovery per agent-hour* = net recovery / logged session hours. About ₹15-16k/h, flat. *Cost per ₹ recovered* is **assumption-driven** (no cost data exists): 1.7 paise per ₹ under `config/assumptions.json`; it is flat because every driver is flat.

## 5. Counterfactual: what if the targeting strategy had not changed? (Part 4)
**Design in `docs/counterfactual_design.md`.** The data contain no discrete switch date: `strategy_version` (legacy, v1, v2, v3) is mixed in every month. So we run the design on the in-data variation: accounts targeted by campaigns on strategy **v2/v3 (treated)** vs **legacy/v1 (control)**, with a pre/post DiD (14 days before vs after the target date), covariate adjustment and a permutation placebo.

In [19]:
t=q('''select t.account_id,t.target_date,cm.strategy_version sv,a.dpd_bucket,a.risk_segment,a.loan_type,
 (select count(*) from clean.payments p where p.account_id=t.account_id and p.payment_status='SUCCESS' and p.event_ts_ist>=t.target_date-interval 14 day and p.event_ts_ist<t.target_date)>0 pre,
 (select count(*) from clean.payments p where p.account_id=t.account_id and p.payment_status='SUCCESS' and p.event_ts_ist>=t.target_date and p.event_ts_ist<t.target_date+interval 14 day)>0 post
 from raw.daily_targeting t join raw.campaigns cm using(campaign_id) join golden.dim_account a using(account_id) where t.target_date between '2026-01-15' and '2026-07-20' ''')
t['pre']=t.pre.astype(int);t['post']=t.post.astype(int);t['treat']=t.sv.isin(['v2','v3']).astype(int)
print(t.groupby('treat')[['pre','post']].agg(['mean','count']).to_string())
L=pd.concat([t.assign(y=t.pre,period=0),t.assign(y=t.post,period=1)])
m=smf.ols('y~treat*period+C(dpd_bucket)+C(risk_segment)+C(loan_type)',data=L).fit(cov_type='cluster',cov_kwds={'groups':L.account_id.astype('category').cat.codes})
lo,hi=m.conf_int().loc['treat:period'];print(f"\nDiD effect on 14-day payment probability: {m.params['treat:period']*100:+.2f} pp, 95% CI [{lo*100:+.2f}, {hi*100:+.2f}] pp, p = {m.pvalues['treat:period']:.2f}")
rng=np.random.default_rng(0);dd=(t.post-t.pre).values;est=[dd[(tt:=rng.permutation(t.treat.values))==1].mean()-dd[tt==0].mean() for _ in range(500)]
obs=dd[t.treat==1].mean()-dd[t.treat==0].mean();print(f'Placebo (500 random re-labellings): sd = {np.std(est)*100:.2f} pp, share |placebo| >= observed = {(np.abs(est)>=abs(obs)).mean():.2f}')
p0=t.post.mean();print(f'Baseline 14-day pay rate {p0:.3f}. Sample needed per arm to detect a +10% relative lift: {2*(1.96+.84)**2*p0*(1-p0)/(p0*.10)**2:,.0f}; +5%: {2*(1.96+.84)**2*p0*(1-p0)/(p0*.05)**2:,.0f}')

         pre          post       
        mean  count   mean  count
treat                            
0     0.0372  19527 0.0383  19527
1     0.0359  18769 0.0365  18769



DiD effect on 14-day payment probability: -0.04 pp, 95% CI [-0.57, +0.49] pp, p = 0.87


Placebo (500 random re-labellings): sd = 0.27 pp, share |placebo| >= observed = 0.88
Baseline 14-day pay rate 0.037. Sample needed per arm to detect a +10% relative lift: 40,336; +5%: 161,343


**Result (Strong evidence of no detectable effect, not evidence of zero):** the DiD is about -0.04 pp with a 95% CI of roughly ±0.5 pp on a 3.7% base, i.e. we can rule out lifts larger than ~+13% relative and the placebo distribution is just as wide. The counterfactual answer is therefore: *recovery would have looked the same.* Key assumptions are listed in the design doc: parallel trends (checked via the pre-period, which is balanced: 3.4-3.9%), no spillovers between accounts, and that `strategy_version` on the campaign is what actually drove targeting (unverifiable here: **Hypothesis**).

## 6. Where should ₹10 Cr go? (Part 4 question 4)
### 6a. What the data say about each lever
Incremental value of a touch = payment probability in the 7 days after the touch minus the probability for random account-days with no regard to touches.

In [20]:
tl=q('''with t as (select account_id,touch_ts,channel,engaged from golden.fact_touch where touch_ts<'2026-08-01'),
x as (select channel,engaged,(exists(select 1 from golden.fact_payment p where p.account_id=t.account_id and p.payment_status='SUCCESS' and p.event_ts_ist>t.touch_ts and p.event_ts_ist<=t.touch_ts+interval 7 day))::int y from t)
select channel,engaged,count(*) n,avg(y) rate from x group by 1,2 order by 1,2''')
bs=q('''with s as (select a.account_id,(date '2026-01-01'+(floor(random()*200))::int*interval 1 day) d from golden.dim_account a,range(10))
select count(*) n,avg((exists(select 1 from golden.fact_payment p where p.account_id=s.account_id and p.payment_status='SUCCESS' and p.event_ts_ist>s.d and p.event_ts_ist<=s.d+interval 7 day))::int) r from s''').iloc[0]
tl['baseline']=bs.r;tl['lift_pp']=(tl.rate-bs.r)*100;se=np.sqrt(tl.rate*(1-tl.rate)/tl.n+bs.r*(1-bs.r)/bs.n);tl['ci_low_pp']=tl.lift_pp-1.96*se*100;tl['ci_high_pp']=tl.lift_pp+1.96*se*100
tl['lift_significant']=(tl.ci_low_pp>0)|(tl.ci_high_pp<0);tl.to_csv('outputs/tables/touch_lift.csv',index=False);tl

,channel,engaged,n,rate,baseline,lift_pp,ci_low_pp,ci_high_pp,lift_significant
0,FIELD,False,12048,0.0198,0.0182,0.1597,-0.0938,0.4133,False
1,FIELD,True,12081,0.0162,0.0182,-0.2016,-0.4319,0.0287,False
2,SMS,False,21792,0.0183,0.0182,0.0069,-0.1774,0.1913,False
3,SMS,True,10738,0.0192,0.0182,0.0944,-0.1694,0.3583,False
4,VOICE,False,64003,0.0181,0.0182,-0.0116,-0.1255,0.1023,False
5,VOICE,True,15912,0.0192,0.0182,0.0928,-0.1256,0.3112,False
6,WHATSAPP,False,19509,0.0170,0.0182,-0.1222,-0.3099,0.0655,False
7,WHATSAPP,True,29136,0.0181,0.0182,-0.0118,-0.1723,0.1487,False


**Strong evidence:** after any touch (engaged or not, on any channel) the 7-day payment probability is 1.6-1.9%, statistically the same as the 1.87% you get for an arbitrary account-day. No channel shows significant positive lift. (Field-engaged shows a slightly *negative* point estimate: treat that as noise.) Nothing observed supports scaling any input. Caveat: touches on the same account are not independent, so these intervals are too narrow, and one of eight comparisons crossing 5% is what chance produces.

### 6b. Recommendation: **Option 4, better borrower targeting, released in stages**
**Why this and not the others.** Options 1, 2, 3, 5, 6 all buy *more touches*, and measured lift per touch is ~0 (6a). Their cost scales linearly with volume. Vendor effect is also nil (contact rate 19.2-21.1% across 15 vendors, p = 0.45), agents are indistinguishable (dispersion 0.99), and 54% of calls are already outside permitted hours, so more calling or AI-voice capacity amplifies a compliance problem. Option 4 is the only one whose cost does not scale with touches, and it carries the thing the business lacks: a **randomized holdout** that finally measures incremental recovery.

**What the data can't support:** the current features (DPD, risk segment, loan type, balance, state) have **no predictive signal for payment** (logit pseudo-R² = 0.0003, p = 0.45). So the ₹10 Cr must buy new *signal* (bureau/alternative data, contactability and best-time data, payment-history features) and the experiment infrastructure, not just a re-ranking of existing columns.

In [21]:
base_annual=q("select avg(net) from (select event_date_ist d,sum(net_amount) net from golden.fact_payment where event_date_ist<'2026-08-01' group by 1)").iloc[0,0]*365
inv=10e7;scen=pd.DataFrame({'scenario':['Downside (data-consistent central case)','Base (assumption)','Upside (assumption)'],'lift_on_annual_recovery':[0.0,0.03,0.08],'probability':[0.50,0.35,0.15]})
scen['incremental_recovery_12m_Cr']=base_annual*scen.lift_on_annual_recovery/1e7
scen['ROI_12m']=scen.incremental_recovery_12m_Cr/10-1
scen['incremental_recovery_36m_Cr']=scen.incremental_recovery_12m_Cr*3
scen['ROI_36m']=scen.incremental_recovery_36m_Cr/10-1
ev12=(scen.probability*scen.incremental_recovery_12m_Cr).sum();ev36=ev12*3
print(f'Annualised baseline net recovery: Rs {base_annual/1e7:,.1f} Cr')
print(f'Break-even lift (value = gross recovered Rs): 12m {inv/base_annual*100:.1f}% of annual recovery ; 36m {inv/3/base_annual*100:.1f}%')
print(f'Probability-weighted incremental recovery: 12m Rs {ev12:.1f} Cr (ROI {ev12/10-1:+.0%}); 36m Rs {ev36:.1f} Cr (ROI {ev36/10-1:+.0%})')
print('Staged plan: release Rs 1.5 Cr for a 90-day holdout; release the remaining Rs 8.5 Cr only if measured lift >= break-even.')
g=0.5;cost=1.5+g*8.5;ev_inc=g*(0.7*scen.incremental_recovery_12m_Cr[1]+0.3*scen.incremental_recovery_12m_Cr[2]);print(f'Staged, 12m: expected spend Rs {cost:.2f} Cr, expected incremental Rs {ev_inc:.2f} Cr, ROI {ev_inc/cost-1:+.0%}; worst-case loss capped at Rs 1.5 Cr instead of Rs 10 Cr')
scen.to_csv('outputs/tables/investment_scenarios.csv',index=False);scen

Annualised baseline net recovery: Rs 202.5 Cr
Break-even lift (value = gross recovered Rs): 12m 4.9% of annual recovery ; 36m 1.6%
Probability-weighted incremental recovery: 12m Rs 4.6 Cr (ROI -54%); 36m Rs 13.7 Cr (ROI +37%)
Staged plan: release Rs 1.5 Cr for a 90-day holdout; release the remaining Rs 8.5 Cr only if measured lift >= break-even.
Staged, 12m: expected spend Rs 5.75 Cr, expected incremental Rs 4.56 Cr, ROI -21%; worst-case loss capped at Rs 1.5 Cr instead of Rs 10 Cr


,scenario,lift_on_annual_recovery,probability,incremental_recovery_12m_Cr,ROI_12m,incremental_recovery_36m_Cr,ROI_36m
0,Downside (data-consistent central case),0.0000,0.5000,0.0000,-1.0000,0.0000,-1.0000
1,Base (assumption),0.0300,0.3500,6.0740,-0.3926,18.2219,0.8222
2,Upside (assumption),0.0800,0.1500,16.1972,0.6197,48.5917,3.8592


**Read this honestly.** Expected incremental recovery, ROI and break-even above come from **assumed** lift scenarios, because the data contain no measurable lift to anchor on. Treat the numbers as a decision frame, not a forecast. Confidence is **low**. The estimates would need to be replaced by the holdout result: ~40,000 accounts per arm detect a +10% relative lift on a 3.7% 14-day payment rate (80% power), ~160,000 for +5%. At 30,000 accounts total, a 90-day rolling account-day randomization (not account-level) is needed. Valuing recovered rupees at 100% is generous: at a 10% commission the revenue break-even is ten times higher, which is why the spend must be gated.

## 7. Evidence classification
| # | Conclusion | Class |
|---|---|---|
| 1 | Reported +11% = raw SUCCESS rows, March vs February | **Fact** (reproduced to 10.99%) |
| 2 | ~10.7 pts of it is 31 vs 28 days; per-day change ≈ -0.1% (p=0.98) | **Fact** |
| 3 | 500 duplicate payment rows (₹3.8 Cr, of which ₹2.6 Cr SUCCESS) inflate gross SUCCESS recovery by ~2.0%, spread evenly across months | **Fact** |
| 4 | Dedupe on `payment_reference` would wrongly delete ~3.8k genuine payments | **Fact** |
| 5 | No portfolio mix shift (DPD, loan, risk, vintage) | **Strong evidence** |
| 6 | No Simpson reversal; uplift appears uniformly in all segments (calendar signature) | **Strong evidence** |
| 7 | `accounts.status` is not a valid filter; 75% of payers are on non-ACTIVE accounts | **Fact** |
| 8 | PTP "kept" rate (~50%) not corroborated by payments (~4.8% verified) | **Fact** |
| 9 | No contact-rate driver survives multiple-testing control; agents indistinguishable | **Strong evidence** |
| 10 | No channel touch lifts 7-day payment probability above baseline | **Strong evidence** |
| 11 | Targeting strategy v2/v3 vs legacy/v1: no detectable effect (CI ±0.5 pp) | **Strong evidence** (of no *large* effect) |
| 12 | 54% of outbound calls are outside 08:00-19:00 IST | **Fact** if timestamps are right |
| 13 | Tenure, language, telephony vendor effects | **Hypothesis** (tenure data conflicted, language absent, vendor tz assumed) |
| 14 | Option 4 ROI and break-even | **Hypothesis** (assumption-driven; low confidence) |